# Reduced Row Echelon Form

*Course notes for **Math for Machine Learning**, C1 · W2 · L2 — "Reduced Row Echelon Form" (DeepLearning.AI).*

[Row echelon form](./C1_W2_L2_row_echelon_form_general.ipynb) clears everything *below* each pivot. **Reduced** row echelon form (RREF) goes one step further and also clears *above* each pivot. A matrix is in **RREF** when:

1. It is already in **row echelon form** (descending staircase, zero rows at the bottom).
2. **Each pivot is a 1.**
3. **Every entry above (and below) a pivot is 0.**

As before, the **rank** is the **number of pivots**. For a non-singular square matrix, the RREF is the **identity**.

In [1]:
import numpy as np
from fractions import Fraction as F

def show(M):
    for row in M:
        print("   [" + "  ".join(f"{str(x):>4}" for x in row) + "]")

def to_ref(M):
    """Row echelon form (leading 1s, zeros below); returns (matrix, pivot columns)."""
    M = [[F(x) for x in row] for row in M]
    rows, cols = len(M), len(M[0])
    pivots, prow = [], 0
    for col in range(cols):
        piv = next((r for r in range(prow, rows) if M[r][col] != 0), None)
        if piv is None:
            continue
        M[prow], M[piv] = M[piv], M[prow]
        M[prow] = [x / M[prow][col] for x in M[prow]]
        for r in range(prow + 1, rows):
            M[r] = [M[r][k] - M[r][col] * M[prow][k] for k in range(cols)]
        pivots.append(col); prow += 1
    return M, pivots

def to_rref(M):
    """Continue from REF, clearing entries ABOVE each pivot (Gauss-Jordan)."""
    M, pivots = to_ref(M)
    for prow, col in enumerate(pivots):
        for r in range(prow):                       # clear above this pivot
            M[r] = [M[r][k] - M[r][col] * M[prow][k] for k in range(len(M[0]))]
    return M, pivots

## 1. From REF to RREF — worked example

Start from a row echelon form:

$$ \begin{bmatrix} 1 & 2 & 3 \\ 0 & 1 & 4 \\ 0 & 0 & 1 \end{bmatrix} $$

Clear the entries **above** the pivots, one pivot at a time:

1. **Subtract 2× row 2 from row 1:** $\;\begin{bmatrix}1&0&-5\\0&1&4\\0&0&1\end{bmatrix}$
2. **Add 5× row 3 to row 1:** $\;\begin{bmatrix}1&0&0\\0&1&4\\0&0&1\end{bmatrix}$
3. **Subtract 4× row 3 from row 2:** $\;\begin{bmatrix}1&0&0\\0&1&0\\0&0&1\end{bmatrix}$

The result is the **identity** — the RREF of a non-singular matrix.

In [2]:
# Reproduce the three clearing steps explicitly.
M = [[F(1), F(2), F(3)], [F(0), F(1), F(4)], [F(0), F(0), F(1)]]

M[0] = [M[0][k] - 2 * M[1][k] for k in range(3)]   # row1 -= 2*row2
print("after step 1:"); show(M)
M[0] = [M[0][k] + 5 * M[2][k] for k in range(3)]   # row1 += 5*row3
print("\nafter step 2:"); show(M)
M[1] = [M[1][k] - 4 * M[2][k] for k in range(3)]   # row2 -= 4*row3
print("\nafter step 3 (RREF = identity):"); show(M)

after step 1:
   [   1     0    -5]
   [   0     1     4]
   [   0     0     1]

after step 2:
   [   1     0     0]
   [   0     1     4]
   [   0     0     1]

after step 3 (RREF = identity):
   [   1     0     0]
   [   0     1     0]
   [   0     0     1]


## 2. RREF in general

A general RREF keeps the staircase but isolates every pivot in its column:

$$
\text{rank 5: } \begin{bmatrix}1&0&0&0&0\\0&1&0&0&0\\0&0&1&0&0\\0&0&0&1&0\\0&0&0&0&1\end{bmatrix}
\qquad
\text{rank 3: } \begin{bmatrix}1&*&0&0&*\\0&0&1&0&*\\0&0&0&1&*\\0&0&0&0&0\\0&0&0&0&0\end{bmatrix}
$$

Each pivot is a 1, alone in its column; `*` are free (non-pivot) columns. The **rank is still the number of pivots** (5 and 3 here). A full-rank square matrix reduces to the identity.

In [3]:
examples = {
    "non-singular [[5,1],[4,-3]]": [[5, 1], [4, -3]],
    "singular     [[1,1],[2,2]]":  [[1, 1], [2, 2]],
    "rank-2 3x3":                  [[1, 1, 1], [1, 2, 1], [1, 1, 2]],
    "rank-2 staircase":           [[1, 1, 1], [1, 1, 2], [1, 1, 3]],
}
for name, A in examples.items():
    rref, pivots = to_rref(A)
    print(f"{name}  ->  rank {len(pivots)} (pivot cols {pivots}):")
    show(rref); print()

non-singular [[5,1],[4,-3]]  ->  rank 2 (pivot cols [0, 1]):
   [   1     0]
   [   0     1]

singular     [[1,1],[2,2]]  ->  rank 1 (pivot cols [0]):
   [   1     1]
   [   0     0]

rank-2 3x3  ->  rank 3 (pivot cols [0, 1, 2]):
   [   1     0     0]
   [   0     1     0]
   [   0     0     1]

rank-2 staircase  ->  rank 2 (pivot cols [0, 2]):
   [   1     1     0]
   [   0     0     1]
   [   0     0     0]



## 3. Why RREF is useful

RREF is the **fully solved** form of a system: each pivot variable is read off directly. For the non-singular system $\begin{bmatrix}5&1\\4&-3\end{bmatrix}$ with solution $a=3, b=2$, augmenting with the constants and reducing gives $\begin{bmatrix}1&0&\mid&3\\0&1&\mid&2\end{bmatrix}$ — the answer appears in the last column.

In [4]:
# Augmented matrix [A | b] for  5a + b = 17,  4a - 3b = 6.
aug = [[5, 1, 17], [4, -3, 6]]
rref, pivots = to_rref(aug)
print("RREF of [A | b]:"); show(rref)
print("\n-> a =", rref[0][-1], ", b =", rref[1][-1])

RREF of [A | b]:
   [   1     0     3]
   [   0     1     2]

-> a = 3 , b = 2


## 4. Takeaways

- **Reduced row echelon form (RREF)** = row echelon form **plus** every pivot is a 1 and is the **only** non-zero entry in its column (zeros above *and* below).
- Reached by continuing elimination **upward** (Gauss–Jordan) after REF.
- **Rank = number of pivots** (unchanged); a non-singular square matrix has RREF equal to the **identity**.
- Augmenting with the constants and reducing to RREF reads the **solution straight off** the last column.

*Companion:* [row echelon form in general](./C1_W2_L2_row_echelon_form_general.ipynb) · [systems to matrices](./C1_W2_L1_systems_to_matrices.ipynb).